# 从 Transformer 到 Mini-GPT

> **前情回顾**：上一章你已经手写了 Self-Attention、Multi-Head 和 Transformer Block。
> **本章目标**：跟着 GPT-2 的结构图，把 Block 堆成一个可以输出 logits 的 Mini-GPT，并对照 Karpathy/nanoGPT 的写法。


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)


## 1. 学习地图

这章只回答一个问题：**已经有 Transformer Block 了，怎么变成 GPT？**

你可以把 GPT 想成一条流水线：

```text
Token IDs
  ↓
Token Embedding + Position Embedding
  ↓
Transformer Block × N
  ↓
LayerNorm
  ↓
Linear
  ↓
logits：每个位置预测下一个 token
```

重点不是背名字，而是盯住 shape：

```text
[batch, seq] → [batch, seq, d_model] → [batch, seq, vocab_size]
```


## 2. 先复用 Block

为了让本 Notebook 单独运行，我们先把上一章的三个零件放在这里。

你先不用重新理解每一行，先记住它们的职责：

1. `MultiHeadAttention`：让 token 互相看，但不能看未来。
2. `FeedForward`：每个 token 自己过一个小网络。
3. `TransformerBlock`：把 Attention、FFN、Residual、LayerNorm 串起来。


In [2]:
class MultiHeadAttention(nn.Module):
    """
    多头因果自注意力
    
    参数:
        d_model: 输入/输出维度
        num_heads: 注意力头数
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # Q、K、V 的线性变换（把 num_heads 个头合并到矩阵操作里）
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        
        # 输出投影
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        """
        输入: x shape = [batch, seq_len, d_model]
        输出:   shape = [batch, seq_len, d_model]
        """
        batch_size, seq_len, _ = x.shape
        
        # 1. 线性变换 + 拆成多头
        #    [batch, seq_len, d_model] → [batch, num_heads, seq_len, d_k]
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. 注意力分数: Q @ K^T
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # 3. Mask（把未来的位置设为 -inf）
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # 4. Softmax
        weights = F.softmax(scores, dim=-1)
        
        # 5. 加权求和
        attn_output = weights @ V  # [batch, num_heads, seq_len, d_k]
        
        # 6. 拼回头并投影
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(attn_output)


In [3]:
class FeedForward(nn.Module):
    """FFN：两层全连接，先扩 4 倍再压回"""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class TransformerBlock(nn.Module):
    """一个 Transformer 解码器层：Attention + FFN，各带残差 + LayerNorm"""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        x = self.norm1(x + self.attention(x, mask))  # Attention + 残差 + Norm
        x = self.norm2(x + self.ffn(x))              # FFN + 残差 + Norm
        return x


## 3. 跟着结构图手写 MiniGPT

现在开始搭整台模型。

第一步：Token ID 只是整数，不能直接送进神经网络，所以要查 `Embedding` 表。

第二步：同一个 token 出现在第 1 个位置和第 10 个位置，意思可能不同，所以要加 Position 信息。

第三步：经过多层 Transformer Block。

最后一步：把 hidden state 投影到词表大小，得到 logits。


In [4]:
# 复用 Part 3 的位置编码
def get_sinusoidal_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class MiniGPT(nn.Module):
    """Mini GPT: Embedding → N×TransformerBlock → LayerNorm → 投影到词表"""
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=128):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        pe = get_sinusoidal_encoding(max_seq_len, d_model)
        self.register_buffer('pe', pe)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)  # 投影到词表 → logits
    
    def forward(self, x):
        # x: [batch, seq_len]  token IDs
        batch_size, seq_len = x.shape
        x = self.token_emb(x) + self.pe[:seq_len, :]          # Embedding + Position
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        mask = mask.view(1, 1, seq_len, seq_len)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_final(x)
        return self.lm_head(x)  # [batch, seq_len, vocab_size]

## 4. 跑一次完整前向传播

模型现在还没训练，所以输出不聪明。

但我们先不关心答案好不好，只看数据有没有沿着结构图正确流动。


In [5]:
# 测试 MiniGPT
vocab_size = 30
model = MiniGPT(vocab_size=vocab_size, d_model=64, num_heads=4, num_layers=2)

total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}, 可训练: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# 前向传播: batch=2, seq_len=8
batch_input = torch.randint(0, vocab_size, (2, 8))
logits = model(batch_input)
print(f"\n输入: {batch_input.shape}  →  输出: {logits.shape}")
print(f"输出 = [batch=2, seq=8, vocab={vocab_size}] → 每个位置对每个词的 logits")
print(f"logits[0, 7, :5]: {logits[0, 7, :5].tolist()}  ← 样本 0 位置 7 预测下一 token 的分数")

总参数量: 103,966, 可训练: 103,966

输入: torch.Size([2, 8])  →  输出: torch.Size([2, 8, 30])
输出 = [batch=2, seq=8, vocab=30] → 每个位置对每个词的 logits
logits[0, 7, :5]: [1.4079980850219727, -0.3056167960166931, -1.065438985824585, 0.11685330420732498, 0.531089186668396]  ← 样本 0 位置 7 预测下一 token 的分数


In [6]:
# 跟着结构图跑一次：每一步都打印 shape
vocab_size = 30
model = MiniGPT(vocab_size=vocab_size, d_model=32, num_heads=4, num_layers=2, max_seq_len=16)
idx = torch.randint(0, vocab_size, (2, 6))

with torch.no_grad():
    token_vec = model.token_emb(idx)
    pos_vec = model.pe[:idx.shape[1], :]
    x = token_vec + pos_vec
    print(f"1. token + position: {x.shape}")

    mask = torch.tril(torch.ones(idx.shape[1], idx.shape[1]))
    mask = mask.view(1, 1, idx.shape[1], idx.shape[1])

    for block_id, block in enumerate(model.blocks, start=1):
        x = block(x, mask)
        print(f"2.{block_id} 经过 TransformerBlock: {x.shape}")

    x = model.ln_final(x)
    logits = model.lm_head(x)
    print(f"3. logits: {logits.shape}")

print("关键观察：中间一直是 d_model 维，只有最后一步才变成 vocab_size。")


1. token + position: torch.Size([2, 6, 32])
2.1 经过 TransformerBlock: torch.Size([2, 6, 32])
2.2 经过 TransformerBlock: torch.Size([2, 6, 32])
3. logits: torch.Size([2, 6, 30])
关键观察：中间一直是 d_model 维，只有最后一步才变成 vocab_size。


## 5. logits 是什么？

`logits` 是模型最后输出的原始分数。

如果 vocab 有 30 个 token，那么每个位置都会输出 30 个分数：

```text
logits[batch, 位置, token_id]
```

分数越高，模型越觉得这个 token 适合作为“下一个 token”。

训练时，我们会用 Cross-Entropy 让正确答案的分数变高。生成时，我们会对 logits 做 softmax，再抽样或取最大值。


## 6. 对照 Karpathy/nanoGPT

Karpathy 的 `nanoGPT` / `minGPT` 更接近真实 GPT-2 工程写法。

但你读代码时不要慌，先找这条主线：

```text
GPT.forward
  → token embedding / position embedding
  → for block in transformer blocks
  → final layer norm
  → lm_head 得到 logits
```

名字会变，主线不会变。


In [7]:
# 用可运行的小表，把教学版名字和 Karpathy/nanoGPT 的名字对上
karpathy_map = [
    ("MiniGPT", "GPT", "整台模型：embedding、blocks、norm、lm_head"),
    ("TransformerBlock", "Block", "一层 Attention + MLP"),
    ("MultiHeadAttention", "CausalSelfAttention", "带 causal mask 的多头自注意力"),
    ("FeedForward", "MLP", "每个 token 独立经过的小网络"),
    ("token_emb", "transformer.wte", "token embedding table"),
    ("pe", "transformer.wpe", "position embedding table"),
    ("lm_head", "lm_head", "把 hidden state 变成 vocab logits"),
]

print("教学版  →  Karpathy/nanoGPT  →  作用")
for ours, karpathy, meaning in karpathy_map:
    print(f"{ours:20s} -> {karpathy:24s} | {meaning}")

print()
print("关键观察：工程代码名字更短、更快，但主线还是这条：")
print("token + position -> blocks -> final norm -> lm_head -> logits")


教学版  →  Karpathy/nanoGPT  →  作用
MiniGPT              -> GPT                      | 整台模型：embedding、blocks、norm、lm_head
TransformerBlock     -> Block                    | 一层 Attention + MLP
MultiHeadAttention   -> CausalSelfAttention      | 带 causal mask 的多头自注意力
FeedForward          -> MLP                      | 每个 token 独立经过的小网络
token_emb            -> transformer.wte          | token embedding table
pe                   -> transformer.wpe          | position embedding table
lm_head              -> lm_head                  | 把 hidden state 变成 vocab logits

关键观察：工程代码名字更短、更快，但主线还是这条：
token + position -> blocks -> final norm -> lm_head -> logits


### GPT-2 的 Position 写法

上一章和刚才的 `MiniGPT` 用的是 sinusoidal position encoding：靠公式生成位置向量。

GPT-2 / nanoGPT 常用另一种方式：**position embedding table**。

也就是：位置 0、位置 1、位置 2……每个位置都有一行可训练向量。


In [8]:
# GPT-2 / nanoGPT 更常见的 position 写法：不是公式，而是可学习表
class GPT2StylePositionDemo(nn.Module):
    """只演示 token embedding + position embedding 这一步"""
    def __init__(self, vocab_size, block_size, d_model):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model)      # token embedding
        self.wpe = nn.Embedding(block_size, d_model)      # position embedding

    def forward(self, idx):
        batch_size, seq_len = idx.shape
        pos = torch.arange(seq_len, device=idx.device)
        return self.wte(idx) + self.wpe(pos)


torch.manual_seed(42)
demo = GPT2StylePositionDemo(vocab_size=30, block_size=16, d_model=8)
idx = torch.randint(0, 30, (2, 6))
x = demo(idx)

print(f"输入 token IDs: {idx.shape}")
print(f"输出向量:      {x.shape}")
print("关键观察：GPT-2 的 wte/wpe 加起来以后，形状还是 [batch, seq, d_model]")


输入 token IDs: torch.Size([2, 6])
输出向量:      torch.Size([2, 6, 8])
关键观察：GPT-2 的 wte/wpe 加起来以后，形状还是 [batch, seq, d_model]


## 7. Special Tokens

Mini-GPT 会输出 logits，但它还需要一些“边界符号”。

比如：哪里开始、哪里结束、哪里是 padding、哪里是思考草稿。

```text
<BOS>       begin of sequence，告诉模型：这里开始
<EOS>       end of sequence，告诉模型：生成到这里可以停
<PAD>       padding，告诉模型：这个位置只是补齐，不是真内容
<think>     thinking start，告诉模型：下面进入思考区间
</think>    thinking end，告诉模型：思考区间结束，下面给最终回答
```

`<think>` 不会让模型自动变聪明。它只是一个符号。

模型要学会用它，训练数据里必须反复出现这种格式。


In [9]:
# 演示：给 Mini-GPT 的词表加入新的 special tokens
base_vocab = {
    "用户": 0,
    "助手": 1,
    "答案": 2,
    "357": 3,
    "289": 4,
    "103173": 5,
}

special_tokens = ["<BOS>", "<EOS>", "<PAD>", "<think>", "</think>"]

vocab = base_vocab.copy()
for token in special_tokens:
    if token not in vocab:
        vocab[token] = len(vocab)

print("新增后的 special token ID:")
for token in special_tokens:
    print(f"  {token:8s} -> {vocab[token]}")

print()
print("关键观察：<think> 和 </think> 现在有独立 ID，")
print("模型才能把它们当成边界符号，而不是普通文字碎片。")

新增后的 special token ID:
  <BOS>    -> 6
  <EOS>    -> 7
  <PAD>    -> 8
  <think>  -> 9
  </think> -> 10

关键观察：<think> 和 </think> 现在有独立 ID，
模型才能把它们当成边界符号，而不是普通文字碎片。


In [10]:
# 演示：一条带 <think> 的训练样本长什么样
sample_tokens = [
    "<BOS>",
    "用户",
    "357",
    "289",
    "助手",
    "<think>",
    "357",
    "289",
    "103173",
    "</think>",
    "答案",
    "103173",
    "<EOS>",
]

sample_ids = [vocab[token] for token in sample_tokens]

print("训练样本 token:")
print(sample_tokens)
print()
print("训练样本 ID:")
print(sample_ids)
print()
print("关键观察：训练数据里出现这种格式，模型才会学会什么时候开始想、什么时候结束想。")

训练样本 token:
['<BOS>', '用户', '357', '289', '助手', '<think>', '357', '289', '103173', '</think>', '答案', '103173', '<EOS>']

训练样本 ID:
[6, 0, 3, 4, 1, 9, 3, 4, 5, 10, 2, 5, 7]

关键观察：训练数据里出现这种格式，模型才会学会什么时候开始想、什么时候结束想。


新增 special tokens 后，词表变大了。

词表一变大，Embedding 表也要变大，因为每个 token 都要有自己的向量。


In [11]:
# 用小矩阵演示：新增 token 后，Embedding 为什么要扩容
old_vocab_size = len(base_vocab)
new_vocab_size = len(vocab)
d_model = 8

torch.manual_seed(42)
old_embedding = nn.Embedding(old_vocab_size, d_model)
new_embedding = nn.Embedding(new_vocab_size, d_model)

# 旧 token 的向量复制过去，新 token 的向量保留随机初始化，后续训练会更新它们
with torch.no_grad():
    new_embedding.weight[:old_vocab_size] = old_embedding.weight

print(f"旧词表大小: {old_vocab_size}")
print(f"新词表大小: {new_vocab_size}")
print(f"Embedding 形状: {tuple(new_embedding.weight.shape)}")
print()
print("关键观察：新增 5 个 special tokens 后，Embedding 也多了 5 行。")
print("这些新行要靠后续训练，才会学到 <think>、</think> 的真正用途。")

旧词表大小: 6
新词表大小: 11
Embedding 形状: (11, 8)

关键观察：新增 5 个 special tokens 后，Embedding 也多了 5 行。
这些新行要靠后续训练，才会学到 <think>、</think> 的真正用途。


## 小结

确认你已经懂了这些：

1. ✅ GPT = Embedding + Position + 多层 Transformer Block + 输出投影
2. ✅ 中间 hidden state 的形状通常保持 `[batch, seq, d_model]`
3. ✅ 最后 `lm_head` 才把维度变成 `vocab_size`
4. ✅ logits 是预测下一个 token 的原始分数
5. ✅ Karpathy/nanoGPT 的名字更工程化，但骨架和我们手写版一致
6. ✅ special tokens 需要独立 ID，也需要扩容 Embedding

下一步：训练。Loss 到底怎么算，模型怎么从乱猜变得会接话。


---

## 本章作业：Mini-GPT

这题检查你是否理解生成时常见的 temperature。

> **使用 AI 的注意事项**：可以让 AI 解释 temperature 对概率分布的影响，但请自己写出缩放 logits 的那一行。


### 作业 3：现代用法 —— temperature 调整 logits

现代 LLM 生成时经常用 temperature 控制随机性。temperature 越小，分布越尖锐；越大，分布越平。

**小提示**：常见写法是 `scaled_logits = logits / temperature`。


In [ ]:
# 作业 3：temperature 缩放 logits 填空
import torch

logits = torch.tensor([1.0, 2.0, 3.0])
temperature = 0.5

# TODO：用 temperature 缩放 logits
scaled_logits = """在这里缩放 logits"""

assert not isinstance(scaled_logits, str), "请先替换三引号里的占位内容"
assert torch.allclose(scaled_logits, torch.tensor([2.0, 4.0, 6.0])), scaled_logits
print("✅ 作业 3 通过：temperature < 1 会让 logits 差距变大，分布更尖锐")
